In [ ]:
# ==========================================
# 📚 Urdu + Arabic PDF → OCR (if needed) → Clean → Stem → Chunk → Embed → FAISS
# ==========================================

# ---- Imports ----
import os, re, fitz, cv2, warnings
import numpy as np
import pandas as pd
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from sentence_transformers import SentenceTransformer
import faiss
from tqdm import tqdm
from paddleocr import PaddleOCR

# ---- Setup ----
warnings.filterwarnings("ignore", category=UserWarning)
nltk.download('stopwords')

pdf_folder = "./sarf-books"
output_folder = "cleaned_sarf_books"
os.makedirs(output_folder, exist_ok=True)

# ---- Urdu + Arabic Stopwords ----
urdu_stopwords = set(stopwords.words('arabic'))
custom_urdu = {'ہے','ہیں','تھا','تھی','کے','کی','کا','میں','سے','پر','اور','کو','نے','یہ','وہ','ہم','تم','کر','گیا','گئی'}
urdu_stopwords.update(custom_urdu)
arabic_stemmer = ISRIStemmer()

# ==========================================
# Step 1️⃣: OCR Setup (Arabic + Urdu)
# ==========================================
ocr = PaddleOCR(lang='ar', use_angle_cls=True, rec_algorithm='SVTR_LCNet')
print("✅ PaddleOCR (Arabic/Urdu) initialized successfully!")

# ==========================================
# Step 2️⃣: Helper + Preprocessing Functions
# ==========================================

def clean_text(text):
    text = re.sub(r'[A-Za-z0-9]', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return re.findall(r'[\u0600-\u06FF]+', text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in urdu_stopwords]

def urdu_stemmer(word):
    suffixes = ['وں','یں','ات','اتوں','یاں','یوں','گی','گا','ی','ے','ا']
    for suf in suffixes:
        if word.endswith(suf) and len(word) > len(suf)+2:
            return word[:-len(suf)]
    return word

def stem_words(tokens):
    stemmed = []
    for t in tokens:
        try:
            s = arabic_stemmer.stem(t)
        except:
            s = urdu_stemmer(t)
        stemmed.append(s)
    return stemmed

def normalize_arabic_text(text):
    text = re.sub("[ـ]+", "", text)                              # Tatweel
    text = re.sub("[إأٱآا]", "ا", text)                         # Alef
    text = re.sub("[يى]", "ی", text)                            # Yeh
    text = re.sub("[هة]", "ہ", text)                            # Heh
    text = re.sub("[\u0617-\u061A\u064B-\u0652]", "", text)     # Diacritics
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_image(image_bytes):
    img = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(img, cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    gray = cv2.equalizeHist(gray)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh

# ==========================================
# Step 3️⃣: PDF Text Extraction (OCR fallback)
# ==========================================
def extract_text_from_pdf(file_path):
    text = ""
    with fitz.open(file_path) as pdf:
        for page in pdf:
            page_text = page.get_text("text")
            if len(page_text.strip()) < 30:  # Low text → scanned
                img = page.get_pixmap(dpi=300)
                img_bytes = img.tobytes("png")
                gray = preprocess_image(img_bytes)

                result = ocr.ocr(gray, cls=True)
                if result and result[0]:
                    page_text = " ".join([line[1][0] for line in result[0]])
                else:
                    page_text = ""
            text += page_text + "\n"
    return text

# ==========================================
# Step 4️⃣: Process PDFs and Save Cleaned Text
# ==========================================
records = []
all_text = ""

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

for file_name in tqdm(pdf_files, desc="📘 Processing PDFs"):
    file_path = os.path.join(pdf_folder, file_name)
    text = extract_text_from_pdf(file_path)
    normalized = normalize_arabic_text(text)
    cleaned = clean_text(normalized)
    tokens = tokenize(cleaned)
    filtered = remove_stopwords(tokens)
    stemmed = stem_words(filtered)
    final_text = " ".join(stemmed)

    out_path = os.path.join(output_folder, f"stemmed_{file_name.replace('.pdf','.txt')}")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    records.append({"book": file_name, "text": final_text})
    all_text += final_text + " "

rag_df = pd.DataFrame(records)
rag_df.to_csv("RAG_ready_books.csv", index=False, encoding="utf-8-sig")
print("\n✅ Saved combined dataset: RAG_ready_books.csv")

# ==========================================
# Step 5️⃣: Split Text into Chunks
# ==========================================
def split_into_chunks(text, max_words=200):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

chunks = []
for _, row in rag_df.iterrows():
    for chunk in split_into_chunks(row["text"], max_words=200):
        chunks.append({"book": row["book"], "chunk": chunk})

chunks_df = pd.DataFrame(chunks)
chunks_df.to_csv("RAG_chunks.csv", index=False, encoding="utf-8-sig")
print(f"✅ Created {len(chunks_df)} text chunks")

# ==========================================
# Step 6️⃣: Embeddings + FAISS Index
# ==========================================
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("\n🔹 Generating embeddings (may take time)...")

embeddings = model.encode(chunks_df["chunk"].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

faiss.write_index(index, "urdu_arabic_books.index")
print("\n✅ FAISS index created and saved as 'urdu_arabic_books.index'")
print(f"Total vectors stored: {index.ntotal}")






c:\Users\ASC\miniconda3\envs\myproject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025/11/02 14:29:43] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\ASC/.paddleocr/whl\\det\\ml\\Multilingual_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\ASC/.paddleocr/whl\\rec\\arabic\\arabic_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ PaddleOCR (Arabic/Urdu) initialized successfully!


📘 Processing PDFs:   0%|          | 0/1 [00:00<?, ?it/s]

[2025/11/02 14:29:53] ppocr DEBUG: dt_boxes num : 8, elapsed : 2.0184333324432373
[2025/11/02 14:29:53] ppocr DEBUG: cls num  : 8, elapsed : 0.3014345169067383
[2025/11/02 14:29:57] ppocr DEBUG: rec_res num  : 8, elapsed : 4.0409605503082275
[2025/11/02 14:30:00] ppocr DEBUG: dt_boxes num : 7, elapsed : 1.7952525615692139
[2025/11/02 14:30:00] ppocr DEBUG: cls num  : 7, elapsed : 0.3529529571533203
[2025/11/02 14:30:05] ppocr DEBUG: rec_res num  : 7, elapsed : 4.537318229675293
[2025/11/02 14:30:08] ppocr DEBUG: dt_boxes num : 12, elapsed : 1.774597406387329
[2025/11/02 14:30:08] ppocr DEBUG: cls num  : 12, elapsed : 0.3620302677154541
[2025/11/02 14:30:15] ppocr DEBUG: rec_res num  : 12, elapsed : 6.626858949661255
[2025/11/02 14:30:18] ppocr DEBUG: dt_boxes num : 60, elapsed : 1.8339216709136963
[2025/11/02 14:30:20] ppocr DEBUG: cls num  : 60, elapsed : 2.0070078372955322
[2025/11/02 14:30:50] ppocr DEBUG: rec_res num  : 60, elapsed : 30.59894895553589
[2025/11/02 14:30:54] ppocr DE

📘 Processing PDFs: 100%|██████████| 1/1 [4:24:24<00:00, 15864.95s/it]



✅ Saved combined dataset: RAG_ready_books.csv
✅ Created 48 text chunks

🔹 Generating embeddings (may take time)...


Batches: 100%|██████████| 2/2 [00:26<00:00, 13.03s/it]


✅ FAISS index created and saved as 'urdu_arabic_books.index'
Total vectors stored: 48

🔍 Example RAG Search Results:

Result 1:
یفنای نانزرفنال زفنال رفاحفم عہب لعم ہانرل مح زیخ یفلیال یفنئال عدبالل نزرفلیال زفذیال حدیال بیر دحو فئل یفلال رذیل زرلیال بفر دیش ذرفنیال رفدیل زفلال ئیل ئیل زفلیال ہركد ذفئال ذیفئال خدئال ئال زفئال نزرفئنال بئافخرحا یفئال قنل نتارفنای زرفنال صمی نانیفنال یفلال نانزفئیال نزفلییال بفش یفذاال ذال عؤا...

Result 2:
یشم ہ ثتوملاو سما ہ فرصارییت با ركدم ءادمال عفاب كفا دمی دمو دمم دم مم دم دم وہف ددا دمیذ ندمبلدئمیل دمیل می دمیل دمیل ننی دم دم ہیت دمت دید دنل دش دیت دیل دل رمل یہنلاو ددمال دہمیال دمیال دمیال دیمئال دیل ذدمیالدمالدمالدمتال ذدمتال ذمم ہنم رظلاو دمیال دئمیال ددمال ننر دنبج كنن دمم زیتم تمم جاخمل جج...

Result 3:
ینمی ہنم نمو نو نمی ہنم یال ردل فكل عنمف رقل باعلاكام عفبب رینرم ہاناقی نقب نیا قمقؤم كدف ہنمرمالا قوا ننقونقولنقوالنقوال قول نوا قول ہنم ہنلاو فمل راسال حج عتف ببب ریصفرم ریسناہمالا سنی ررس رصری رفو فوج لئء زكرورف خحع ٢ نور ءایواو ررت شزؤؤف

In [5]:
def search_query(query, top_k=3):
    q_emb = embedder.encode([query])
    q_np = np.array(q_emb, dtype="float32")
    
    # Ensure correct shape: (1, d)
    if q_np.ndim == 1:
        q_np = q_np.reshape(1, -1)

    # Optional: normalize if your index is L2-normalized
    faiss.normalize_L2(q_np)

    D, I = index.search(q_np, top_k)
    results = []
    for idx in I[0]:
        if idx < len(chunks_df):
            results.append({
                "book": chunks_df.iloc[idx]["book"],
                "chunk": chunks_df.iloc[idx]["chunk"]
            })
    return results
